In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import GroupKFold

df = pd.read_csv('data/processed/laps_2023_pitwindow.csv')

feature_cols = [
    'TyreWearRatio', 'LapsSinceLastPit', 'LapNumberNorm',
    'GridPitFraction', 'Stint', 'Position'
]
cat_cols = ['Compound']  # maybe 'Driver' too, but be careful for generalisation

X = df[feature_cols + cat_cols]
y = df['pit_within_3_laps']
groups = df['RaceId']   # e.g. "2023-Bahrain", "2023-Austria"

preprocess = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
        ('num', 'passthrough', feature_cols)
    ]
)

logreg = LogisticRegression(
    max_iter=1000,
    class_weight='balanced'
)

model = Pipeline([
    ('preprocess', preprocess),
    ('clf', logreg)
])

gkf = GroupKFold(n_splits=5)
aucs = []

for train_idx, test_idx in gkf.split(X, y, groups):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    model.fit(X_train, y_train)
    y_proba = model.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, y_proba)
    aucs.append(auc)

print("Race-wise CV AUCs:", aucs)
print("Mean AUC:", sum(aucs)/len(aucs))
